# 🗄️ SQL Interview Preparation Notebook
## IN2IT Technologies — Python Developer (Fresher)
### 📅 Interview Date: March 12th | 📍 Noida | 💰 2.8 LPA

---

> **How to use this notebook:**
> This notebook uses **SQLite** — it runs entirely in Python with no extra installation needed.
> Every code cell creates real tables, inserts real data, and runs real SQL queries.
> **Run every cell from top to bottom** so the database is set up before you query it.

---

### 📚 Table of Contents
1. [Database Setup — Create Tables & Insert Data](#setup)
2. [SQL Command Categories — DDL, DML, DQL, DCL, TCL](#categories)
3. [SELECT — Querying Data](#select)
4. [WHERE — Filtering Rows](#where)
5. [ORDER BY, LIMIT, DISTINCT](#orderby)
6. [Aggregate Functions](#aggregate)
7. [GROUP BY and HAVING](#groupby)
8. [JOINS — Combining Tables](#joins)
9. [Subqueries](#subqueries)
10. [String, Date & Math Functions](#functions)
11. [Keys, Constraints & Indexes](#keys)
12. [Normalization](#normalization)
13. [Transactions](#transactions)
14. [Views & CTEs](#views)
15. [Classic Interview Coding Questions](#coding)
16. [Top 20 Interview Q&A](#qa)
17. [Final Tips & Query Execution Order](#tips)

---
<a id='setup'></a>
## ⚙️ DATABASE SETUP — Run This First!

We use Python's built-in `sqlite3` module — no installation needed.
All tables are created in memory. Run this cell before any other cell.

In [0]:
import sqlite3

# Connect to an in-memory database (resets every time you run)
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row   # allows column name access
cur  = conn.cursor()

# Helper function to print query results neatly
def run(sql, params=()):
    try:
        cur.execute(sql, params)
        rows = cur.fetchall()
        if rows:
            cols = [d[0] for d in cur.description]
            col_widths = [max(len(str(c)), max((len(str(r[i])) for r in rows), default=0))
                          for i, c in enumerate(cols)]
            header = " | ".join(str(c).ljust(w) for c, w in zip(cols, col_widths))
            sep    = "-+-".join("-" * w for w in col_widths)
            print(header)
            print(sep)
            for row in rows:
                print(" | ".join(str(row[i]).ljust(col_widths[i]) for i in range(len(cols))))
            print(f"\n({len(rows)} row{'s' if len(rows)!=1 else ''})")
        else:
            print("Query executed. Rows affected:", cur.rowcount)
        conn.commit()
    except Exception as e:
        print(f"ERROR: {e}")

print("✅ Database connection ready!")
print("✅ run() helper function loaded!")

In [0]:
# ── CREATE TABLES ─────────────────────────────────────────────────────────────

cur.executescript("""
-- Departments table
CREATE TABLE departments (
    department_id   INTEGER PRIMARY KEY,
    department_name TEXT    NOT NULL UNIQUE,
    location        TEXT    NOT NULL,
    budget          REAL    DEFAULT 0
);

-- Employees table
CREATE TABLE employees (
    employee_id   INTEGER PRIMARY KEY,
    first_name    TEXT    NOT NULL,
    last_name     TEXT    NOT NULL,
    email         TEXT    UNIQUE,
    salary        REAL    NOT NULL DEFAULT 0,
    hire_date     TEXT,
    department_id INTEGER,
    manager_id    INTEGER,
    FOREIGN KEY (department_id) REFERENCES departments(department_id)
);

-- Projects table
CREATE TABLE projects (
    project_id   INTEGER PRIMARY KEY,
    project_name TEXT    NOT NULL,
    department_id INTEGER,
    budget       REAL,
    start_date   TEXT,
    end_date     TEXT
);

-- Employee-Project mapping (many-to-many)
CREATE TABLE emp_projects (
    employee_id INTEGER,
    project_id  INTEGER,
    role        TEXT,
    PRIMARY KEY (employee_id, project_id)
);
""")

print("✅ Tables created: departments, employees, projects, emp_projects")

In [0]:
# ── INSERT SAMPLE DATA ────────────────────────────────────────────────────────

cur.executescript("""
-- Departments
INSERT INTO departments VALUES (10, 'Engineering',  'Noida',       5000000);
INSERT INTO departments VALUES (20, 'Marketing',    'Delhi',       2000000);
INSERT INTO departments VALUES (30, 'HR',           'Gurugram',    1500000);
INSERT INTO departments VALUES (40, 'Finance',      'Noida',       3000000);
INSERT INTO departments VALUES (50, 'Research',     'Bengaluru',   4000000);

-- Employees (manager_id = NULL means they are top-level)
INSERT INTO employees VALUES (1,  'Arjun',   'Sharma',  'arjun@company.com',   85000, '2020-03-15', 10, NULL);
INSERT INTO employees VALUES (2,  'Priya',   'Singh',   'priya@company.com',   72000, '2021-06-01', 10, 1);
INSERT INTO employees VALUES (3,  'Rahul',   'Gupta',   'rahul@company.com',   68000, '2021-08-20', 10, 1);
INSERT INTO employees VALUES (4,  'Sneha',   'Kumar',   'sneha@company.com',   91000, '2019-11-10', 20, NULL);
INSERT INTO employees VALUES (5,  'Vikram',  'Patel',   'vikram@company.com',  55000, '2022-01-05', 20, 4);
INSERT INTO employees VALUES (6,  'Ananya',  'Reddy',   'ananya@company.com',  62000, '2022-03-22', 30, NULL);
INSERT INTO employees VALUES (7,  'Karan',   'Mehta',   'karan@company.com',   48000, '2023-07-11', 30, 6);
INSERT INTO employees VALUES (8,  'Divya',   'Nair',    'divya@company.com',   95000, '2018-05-30', 40, NULL);
INSERT INTO employees VALUES (9,  'Rohan',   'Joshi',   'rohan@company.com',   77000, '2020-09-14', 40, 8);
INSERT INTO employees VALUES (10, 'Meena',   'Pillai',  'meena@company.com',   83000, '2021-12-01', 50, NULL);
INSERT INTO employees VALUES (11, 'Aditya',  'Shah',    'aditya@company.com',  59000, '2023-02-28', 10, 1);
INSERT INTO employees VALUES (12, 'Pooja',   'Verma',   'pooja@company.com',   71000, '2022-10-15', 50, 10);
INSERT INTO employees VALUES (13, 'Suresh',  'Yadav',   'suresh@company.com',  44000, '2023-11-01', 30, 6);
INSERT INTO employees VALUES (14, 'Kavita',  'Mishra',  'kavita@company.com',  66000, '2021-04-18', 20, 4);
INSERT INTO employees VALUES (15, 'Nikhil',  'Tiwari',  NULL,                  52000, '2024-01-10', 10, 1);

-- Projects
INSERT INTO projects VALUES (101, 'Customer Portal',   10, 800000,  '2023-01-01', '2023-12-31');
INSERT INTO projects VALUES (102, 'AI Analytics',      50, 1200000, '2023-03-01', '2024-06-30');
INSERT INTO projects VALUES (103, 'Brand Campaign',    20, 400000,  '2023-06-01', '2023-09-30');
INSERT INTO projects VALUES (104, 'ERP Upgrade',       40, 600000,  '2023-02-01', '2024-01-31');
INSERT INTO projects VALUES (105, 'HR Automation',     30, 300000,  '2024-01-01', '2024-06-30');

-- Employee-Project assignments
INSERT INTO emp_projects VALUES (1,  101, 'Tech Lead');
INSERT INTO emp_projects VALUES (2,  101, 'Developer');
INSERT INTO emp_projects VALUES (3,  101, 'Developer');
INSERT INTO emp_projects VALUES (10, 102, 'Project Manager');
INSERT INTO emp_projects VALUES (12, 102, 'Researcher');
INSERT INTO emp_projects VALUES (4,  103, 'Campaign Lead');
INSERT INTO emp_projects VALUES (14, 103, 'Analyst');
INSERT INTO emp_projects VALUES (8,  104, 'Finance Lead');
INSERT INTO emp_projects VALUES (9,  104, 'Analyst');
INSERT INTO emp_projects VALUES (6,  105, 'HR Lead');
""")

conn.commit()
print("✅ Sample data inserted!")
print("   15 employees | 5 departments | 5 projects | 10 assignments")

In [0]:
# ── VERIFY SETUP ──────────────────────────────────────────────────────────────
print("=== DEPARTMENTS ===")
run("SELECT * FROM departments")

print("\n=== EMPLOYEES ===")
run("SELECT * FROM employees")

---
<a id='categories'></a>
## 📌 LESSON 1 — SQL Command Categories

SQL commands are divided into 5 categories. **Interviewers always ask this.**

| Category | Full Name | Commands | Purpose |
|----------|-----------|----------|---------|
| **DDL** | Data Definition Language | CREATE, ALTER, DROP, TRUNCATE | Define structure |
| **DML** | Data Manipulation Language | INSERT, UPDATE, DELETE | Modify data |
| **DQL** | Data Query Language | SELECT | Retrieve data |
| **DCL** | Data Control Language | GRANT, REVOKE | Permissions |
| **TCL** | Transaction Control Language | COMMIT, ROLLBACK, SAVEPOINT | Transactions |

---
### 💡 Interview Questions
**Q: Difference between DDL and DML?**
DDL changes the **structure** (table design). DML changes the **data** inside tables.

**Q: Difference between DELETE and TRUNCATE?**
- DELETE removes specific rows, can be rolled back, fires triggers
- TRUNCATE removes ALL rows, faster, cannot be rolled back, resets auto-increment

In [0]:
# DDL — Creating and Modifying Structure
# (Our main tables already exist — let's demo ALTER)

# Add a new column to employees
run("ALTER TABLE employees ADD COLUMN phone TEXT")
print("Added 'phone' column to employees\n")

# Verify the new column exists
run("SELECT employee_id, first_name, phone FROM employees LIMIT 3")

In [0]:
# DML — INSERT, UPDATE, DELETE

# INSERT — add a new employee
run("""
INSERT INTO employees (employee_id, first_name, last_name, email, salary, hire_date, department_id)
VALUES (16, 'Test', 'User', 'test@company.com', 35000, '2024-03-01', 10)
""")

# Verify insert
run("SELECT * FROM employees WHERE employee_id = 16")

In [0]:
# UPDATE — modify existing data
run("""
UPDATE employees
SET salary = 38000, phone = '9876543210'
WHERE employee_id = 16
""")

run("SELECT employee_id, first_name, salary, phone FROM employees WHERE employee_id = 16")

In [0]:
# DELETE — remove specific rows
run("DELETE FROM employees WHERE employee_id = 16")

print("After delete:")
run("SELECT COUNT(*) AS total_employees FROM employees")

---
<a id='select'></a>
## 📌 LESSON 2 — SELECT — Querying Data

SELECT is the most used SQL command. It retrieves data from one or more tables.

**Basic syntax:**
```sql
SELECT column1, column2
FROM table_name;
```

Use `*` to select all columns (avoid in production — always name your columns).

In [0]:
# Select all columns
run("SELECT * FROM employees")

In [0]:
# Select specific columns only
run("SELECT first_name, last_name, salary FROM employees")

In [0]:
# Column aliases — rename columns in output using AS
run("""
SELECT
    first_name                    AS "First Name",
    last_name                     AS "Last Name",
    salary                        AS "Monthly Salary (₹)",
    salary * 12                   AS "Annual Salary (₹)",
    UPPER(first_name || ' ' || last_name) AS "Full Name"
FROM employees
""")

In [0]:
# DISTINCT — remove duplicate values
print("All department IDs (with duplicates):")
run("SELECT department_id FROM employees ORDER BY department_id")

print("\nUnique department IDs only:")
run("SELECT DISTINCT department_id FROM employees ORDER BY department_id")

---
<a id='where'></a>
## 📌 LESSON 3 — WHERE — Filtering Rows

WHERE filters rows before they are returned. It uses conditions with operators.

| Operator | Meaning |
|----------|---------|
| `=` | Equal |
| `!=` or `<>` | Not equal |
| `>`, `<`, `>=`, `<=` | Comparison |
| `BETWEEN a AND b` | Inclusive range |
| `IN (a, b, c)` | Match any in list |
| `LIKE 'pattern'` | Pattern match (`%` = any chars, `_` = one char) |
| `IS NULL` | Value is missing |
| `AND`, `OR`, `NOT` | Logical operators |

In [0]:
# Basic comparison
run("SELECT first_name, salary FROM employees WHERE salary > 70000")

In [0]:
# AND — both conditions must be true
run("""
SELECT first_name, salary, department_id
FROM employees
WHERE salary > 60000 AND department_id = 10
""")

In [0]:
# OR — at least one condition must be true
run("""
SELECT first_name, department_id
FROM employees
WHERE department_id = 10 OR department_id = 50
""")

In [0]:
# BETWEEN — inclusive range
run("""
SELECT first_name, salary
FROM employees
WHERE salary BETWEEN 60000 AND 80000
ORDER BY salary
""")

In [0]:
# IN — match any value in a list
run("""
SELECT first_name, department_id
FROM employees
WHERE department_id IN (10, 50)
ORDER BY department_id
""")

In [0]:
# LIKE — pattern matching
# % means any number of any characters
# _ means exactly one character

print("Names starting with 'A':")
run("SELECT first_name FROM employees WHERE first_name LIKE 'A%'")

print("\nNames ending with 'a':")
run("SELECT first_name FROM employees WHERE first_name LIKE '%a'")

print("\nNames containing 'an':")
run("SELECT first_name FROM employees WHERE first_name LIKE '%an%'")

In [0]:
# IS NULL — find missing values
print("Employees with no email:")
run("SELECT first_name, email FROM employees WHERE email IS NULL")

print("\nEmployees with no manager (top-level):")
run("SELECT first_name, manager_id FROM employees WHERE manager_id IS NULL")

print("\nEmployees WITH a manager:")
run("SELECT first_name, manager_id FROM employees WHERE manager_id IS NOT NULL")

---
<a id='orderby'></a>
## 📌 LESSON 4 — ORDER BY, LIMIT, DISTINCT

- **ORDER BY** sorts the result. Default is ascending (ASC). Use DESC for descending.
- **LIMIT** restricts how many rows are returned.
- **DISTINCT** removes duplicate values.

In [0]:
# ORDER BY — ascending (default)
run("SELECT first_name, salary FROM employees ORDER BY salary")

In [0]:
# ORDER BY — descending
run("SELECT first_name, salary FROM employees ORDER BY salary DESC")

In [0]:
# ORDER BY multiple columns
run("""
SELECT first_name, department_id, salary
FROM employees
ORDER BY department_id ASC, salary DESC
""")

In [0]:
# LIMIT — top N rows
print("Top 5 highest paid employees:")
run("""
SELECT first_name, salary
FROM employees
ORDER BY salary DESC
LIMIT 5
""")

In [0]:
# LIMIT with OFFSET — pagination
# OFFSET skips N rows before returning results
print("Employees ranked 6th to 10th by salary:")
run("""
SELECT first_name, salary
FROM employees
ORDER BY salary DESC
LIMIT 5 OFFSET 5
""")

---
<a id='aggregate'></a>
## 📌 LESSON 5 — Aggregate Functions

Aggregate functions compute a **single value** from multiple rows.

| Function | Description |
|----------|-------------|
| `COUNT(*)` | Number of rows |
| `COUNT(col)` | Non-NULL values in column |
| `SUM(col)` | Total of numeric column |
| `AVG(col)` | Average value |
| `MAX(col)` | Highest value |
| `MIN(col)` | Lowest value |

In [0]:
# COUNT — number of rows
print("Total employees:")
run("SELECT COUNT(*) AS total_employees FROM employees")

print("\nEmployees WITH an email (ignores NULL):")
run("SELECT COUNT(email) AS employees_with_email FROM employees")

print("\nEmployees WITHOUT an email:")
run("SELECT COUNT(*) - COUNT(email) AS no_email FROM employees")

In [0]:
# SUM, AVG, MAX, MIN
run("""
SELECT
    COUNT(*)        AS total_employees,
    ROUND(SUM(salary), 2)   AS total_payroll,
    ROUND(AVG(salary), 2)   AS average_salary,
    MAX(salary)     AS highest_salary,
    MIN(salary)     AS lowest_salary
FROM employees
""")

In [0]:
# Aggregate with WHERE — filter before aggregating
print("Stats for Engineering (dept 10) only:")
run("""
SELECT
    COUNT(*)              AS headcount,
    ROUND(AVG(salary), 2) AS avg_salary,
    MAX(salary)           AS max_salary
FROM employees
WHERE department_id = 10
""")

---
<a id='groupby'></a>
## 📌 LESSON 6 — GROUP BY and HAVING

**GROUP BY** groups rows with the same value and applies aggregate functions to each group.
**HAVING** filters those groups (like WHERE but for groups — runs AFTER grouping).

### ⚠️ Key Rule
- Use **WHERE** to filter **rows** (before grouping)
- Use **HAVING** to filter **groups** (after grouping)
- You **cannot** use aggregate functions in WHERE

In [0]:
# GROUP BY — employee count per department
run("""
SELECT
    department_id,
    COUNT(*)              AS employee_count,
    ROUND(AVG(salary), 2) AS avg_salary,
    SUM(salary)           AS total_salary_cost
FROM employees
GROUP BY department_id
ORDER BY avg_salary DESC
""")

In [0]:
# GROUP BY with JOIN to show department names
run("""
SELECT
    d.department_name,
    COUNT(e.employee_id)  AS headcount,
    ROUND(AVG(e.salary), 2) AS avg_salary,
    MAX(e.salary)         AS top_salary
FROM employees e
JOIN departments d ON e.department_id = d.department_id
GROUP BY d.department_id, d.department_name
ORDER BY headcount DESC
""")

In [0]:
# HAVING — filter groups AFTER aggregation

print("Departments with MORE than 2 employees:")
run("""
SELECT department_id, COUNT(*) AS emp_count
FROM employees
GROUP BY department_id
HAVING COUNT(*) > 2
""")

In [0]:
# WHERE + GROUP BY + HAVING together
# WHERE filters rows first, then GROUP BY groups them, then HAVING filters groups

print("Among employees hired after 2020:")
print("Show departments where AVERAGE salary > 65000\n")
run("""
SELECT
    department_id,
    COUNT(*)              AS emp_count,
    ROUND(AVG(salary), 2) AS avg_salary
FROM employees
WHERE hire_date > '2020-01-01'
GROUP BY department_id
HAVING AVG(salary) > 65000
ORDER BY avg_salary DESC
""")

---
<a id='joins'></a>
## 📌 LESSON 7 — JOINS — Combining Tables

JOINs combine rows from two or more tables based on a related column.
This is the **most important SQL topic** for interviews.

```
Table A          Table B
┌─────┐          ┌─────┐
│  1  │──────────│  1  │  ← INNER JOIN: only matched rows
│  2  │──────────│  2  │
│  3  │    ╳     │  4  │  ← 3 is in A but not B
│     │    ╳     │  5  │  ← 4,5 are in B but not A
└─────┘          └─────┘
```

| JOIN Type | Returns |
|-----------|---------|
| INNER JOIN | Only matching rows from both tables |
| LEFT JOIN | All rows from LEFT + matching from right (NULL if no match) |
| RIGHT JOIN | All rows from RIGHT + matching from left (NULL if no match) |
| FULL OUTER JOIN | All rows from both tables |
| SELF JOIN | Table joined with itself |

In [0]:
# INNER JOIN — only employees WITH a valid department
print("INNER JOIN — employees with their department name:")
run("""
SELECT
    e.employee_id,
    e.first_name,
    e.salary,
    d.department_name,
    d.location
FROM employees e
INNER JOIN departments d ON e.department_id = d.department_id
ORDER BY d.department_name, e.first_name
""")

In [0]:
# LEFT JOIN — ALL employees, even those without a department
# (If no match in departments, department columns show NULL)

# First, add an employee with no department to demonstrate
run("INSERT INTO employees (employee_id, first_name, last_name, salary) VALUES (99, 'Nodept', 'Person', 30000)")

print("LEFT JOIN — all employees including those with no department:")
run("""
SELECT
    e.first_name,
    e.salary,
    d.department_name
FROM employees e
LEFT JOIN departments d ON e.department_id = d.department_id
ORDER BY d.department_name
""")

# Clean up
run("DELETE FROM employees WHERE employee_id = 99")

In [0]:
# SELF JOIN — employee with their manager's name
# Join employees table with ITSELF
# e = the employee, m = the manager

print("Self Join — each employee and their manager:")
run("""
SELECT
    e.first_name  AS employee,
    e.salary      AS emp_salary,
    m.first_name  AS manager,
    m.salary      AS mgr_salary
FROM employees e
LEFT JOIN employees m ON e.manager_id = m.employee_id
ORDER BY m.first_name, e.first_name
""")

In [0]:
# Multiple JOINs — employees + departments + projects
run("""
SELECT
    e.first_name     AS employee,
    d.department_name,
    p.project_name,
    ep.role
FROM employees e
JOIN departments d  ON e.department_id = d.department_id
JOIN emp_projects ep ON e.employee_id  = ep.employee_id
JOIN projects p     ON ep.project_id   = p.project_id
ORDER BY d.department_name, e.first_name
""")

In [0]:
# Find departments with NO employees (no match in left table)
# RIGHT JOIN or use LEFT JOIN from departments side

print("Departments with employees:")
run("""
SELECT d.department_name, COUNT(e.employee_id) AS emp_count
FROM departments d
LEFT JOIN employees e ON d.department_id = e.department_id
GROUP BY d.department_id, d.department_name
ORDER BY emp_count DESC
""")

---
<a id='subqueries'></a>
## 📌 LESSON 8 — Subqueries

A subquery is a **query inside another query**. The inner query runs first.

**Types:**
- **Scalar subquery** — returns a single value
- **Row subquery** — returns one row
- **Table subquery** — returns multiple rows (used with IN, EXISTS)
- **Correlated subquery** — references outer query, runs once per row

In [0]:
# Scalar subquery — single value in WHERE
print("Employees earning MORE than the company average salary:")
run("""
SELECT first_name, salary
FROM employees
WHERE salary > (SELECT AVG(salary) FROM employees)
ORDER BY salary DESC
""")

# Show the average for reference
run("SELECT ROUND(AVG(salary), 2) AS company_avg FROM employees")

In [0]:
# Subquery with IN
print("Employees working on projects:")
run("""
SELECT first_name, department_id
FROM employees
WHERE employee_id IN (
    SELECT DISTINCT employee_id FROM emp_projects
)
""")

print("\nEmployees NOT assigned to any project:")
run("""
SELECT first_name
FROM employees
WHERE employee_id NOT IN (
    SELECT DISTINCT employee_id FROM emp_projects
)
""")

In [0]:
# Subquery in FROM clause (inline view / derived table)
print("Department-level stats with a label:")
run("""
SELECT
    dept_stats.department_id,
    dept_stats.avg_salary,
    CASE
        WHEN dept_stats.avg_salary >= 80000 THEN 'High Pay'
        WHEN dept_stats.avg_salary >= 60000 THEN 'Mid Pay'
        ELSE 'Low Pay'
    END AS pay_level
FROM (
    SELECT department_id, ROUND(AVG(salary), 2) AS avg_salary
    FROM employees
    GROUP BY department_id
) AS dept_stats
ORDER BY avg_salary DESC
""")

In [0]:
# Correlated subquery — inner query references outer query
# Runs once for EACH row of the outer query

print("Employees earning more than the average of their OWN department:")
run("""
SELECT e.first_name, e.salary, e.department_id
FROM employees e
WHERE e.salary > (
    SELECT AVG(salary)
    FROM employees
    WHERE department_id = e.department_id   -- references outer e!
)
ORDER BY e.department_id, e.salary DESC
""")

---
<a id='functions'></a>
## 📌 LESSON 9 — String, Math & Date Functions

SQL has many built-in functions for transforming data. These are used heavily in real queries.

In [0]:
# ── STRING FUNCTIONS ─────────────────────────────────────────────────────────
run("""
SELECT
    first_name,
    last_name,
    UPPER(first_name)                         AS upper_name,
    LOWER(last_name)                          AS lower_name,
    LENGTH(first_name)                        AS name_length,
    first_name || ' ' || last_name            AS full_name,
    SUBSTR(first_name, 1, 3)                  AS first_3_chars,
    REPLACE(last_name, 'a', '@')              AS replaced
FROM employees
LIMIT 6
""")

In [0]:
# ── MATH FUNCTIONS ───────────────────────────────────────────────────────────
run("""
SELECT
    first_name,
    salary,
    ROUND(salary / 12, 2)     AS monthly_take_home,
    ROUND(salary * 0.1, 2)    AS bonus_10_percent,
    ABS(salary - 70000)       AS diff_from_70k,
    CAST(salary AS INTEGER)   AS salary_int
FROM employees
LIMIT 6
""")

In [0]:
# ── CASE WHEN (conditional logic — like if/else in SQL) ──────────────────────
# This is very important and asked frequently in interviews!

run("""
SELECT
    first_name,
    salary,
    CASE
        WHEN salary >= 85000 THEN '⭐ Senior'
        WHEN salary >= 65000 THEN '🔵 Mid-level'
        WHEN salary >= 50000 THEN '🟡 Junior'
        ELSE '🔴 Entry'
    END AS level,
    CASE department_id
        WHEN 10 THEN 'Engineering'
        WHEN 20 THEN 'Marketing'
        WHEN 30 THEN 'HR'
        WHEN 40 THEN 'Finance'
        WHEN 50 THEN 'Research'
        ELSE 'Unknown'
    END AS dept_name
FROM employees
ORDER BY salary DESC
""")

In [0]:
# ── DATE FUNCTIONS ───────────────────────────────────────────────────────────
# SQLite uses date() and strftime() for date operations

run("""
SELECT
    first_name,
    hire_date,
    SUBSTR(hire_date, 1, 4)   AS hire_year,
    SUBSTR(hire_date, 6, 2)   AS hire_month,
    CAST(
        (JULIANDAY('now') - JULIANDAY(hire_date)) / 365
    AS INTEGER)               AS years_at_company
FROM employees
WHERE hire_date IS NOT NULL
ORDER BY hire_date
""")

---
<a id='keys'></a>
## 📌 LESSON 10 — Keys & Constraints

### Types of Keys (Must Know for Interview)

| Key | Description |
|-----|-------------|
| **Primary Key** | Uniquely identifies each row. NOT NULL + UNIQUE. One per table. |
| **Foreign Key** | References PK of another table. Enforces referential integrity. |
| **Unique Key** | All values must be different. Allows ONE NULL. |
| **Candidate Key** | Any column that COULD be a PK (unique + not null). |
| **Composite Key** | PK made of two or more columns together. |
| **Super Key** | Any set of columns that uniquely identifies a row. |

### Constraints
Constraints enforce rules on data at the database level.

| Constraint | Rule |
|------------|------|
| `PRIMARY KEY` | Unique + Not Null |
| `FOREIGN KEY` | Must exist in referenced table |
| `NOT NULL` | Must have a value |
| `UNIQUE` | No duplicates allowed |
| `DEFAULT` | Use this value if none provided |
| `CHECK` | Value must satisfy condition |

In [0]:
# Demonstrate constraints in action

# Create a table with various constraints
run("""
CREATE TABLE students (
    student_id   INTEGER PRIMARY KEY,
    name         TEXT    NOT NULL,
    email        TEXT    UNIQUE,
    age          INTEGER CHECK(age >= 18 AND age <= 60),
    grade        TEXT    DEFAULT 'Pending',
    course_id    INTEGER
)
""")

# Insert valid data
run("INSERT INTO students VALUES (1, 'Alice', 'alice@uni.com', 20, 'A', 101)")
run("INSERT INTO students VALUES (2, 'Bob',   'bob@uni.com',   22, 'B', 101)")
run("INSERT INTO students (student_id, name, email, age) VALUES (3, 'Charlie', 'charlie@uni.com', 19)")  # grade defaults to 'Pending'

run("SELECT * FROM students")

In [0]:
# Show constraint violations (these will fail — that's expected!)
print("Testing NOT NULL constraint:")
try:
    cur.execute("INSERT INTO students VALUES (4, NULL, 'x@x.com', 20, 'A', 101)")
    conn.commit()
except Exception as e:
    print(f"✅ NOT NULL constraint caught: {e}")

print("\nTesting UNIQUE constraint:")
try:
    cur.execute("INSERT INTO students VALUES (5, 'Dave', 'alice@uni.com', 21, 'B', 101)")
    conn.commit()
except Exception as e:
    print(f"✅ UNIQUE constraint caught: {e}")

print("\nTesting CHECK constraint (age < 18):")
try:
    cur.execute("INSERT INTO students VALUES (6, 'Eve', 'eve@uni.com', 15, 'A', 101)")
    conn.commit()
except Exception as e:
    print(f"✅ CHECK constraint caught: {e}")

In [0]:
# Indexes — speed up data retrieval
# (SQLite automatically creates index on PRIMARY KEY)

# Create an index on salary for faster salary-based queries
run("CREATE INDEX idx_emp_salary ON employees(salary)")
run("CREATE INDEX idx_emp_dept   ON employees(department_id)")

print("✅ Indexes created on employees(salary) and employees(department_id)")
print()
print("Indexes in the database:")
run("SELECT name, tbl_name FROM sqlite_master WHERE type='index'")

### 💡 Interview Questions — Keys & Constraints
**Q: Can a table have multiple primary keys?**
A: No. A table can have only ONE primary key (though it can be composite — made of multiple columns).

**Q: Can a foreign key be NULL?**
A: Yes. A NULL foreign key means "no relationship" — the row simply isn't associated with any parent row.

**Q: What is referential integrity?**
A: The guarantee that a foreign key value either points to an existing primary key value, or is NULL. You cannot have a "dangling" reference.

---
<a id='normalization'></a>
## 📌 LESSON 11 — Normalization

Normalization is the process of organizing a database to **reduce data redundancy** and **improve data integrity**.

### Normal Forms

| Form | Rule |
|------|------|
| **1NF** | Atomic values only. No repeating groups. Each cell has one value. |
| **2NF** | 1NF + No partial dependency. Every non-key column depends on the WHOLE primary key. |
| **3NF** | 2NF + No transitive dependency. Non-key columns depend ONLY on the primary key, not on each other. |

### Example of Normalization

In [0]:
# ── UNNORMALIZED table (violates all 3 forms) ────────────────────────────────
run("""
CREATE TABLE unnormalized_orders (
    order_id    INTEGER,
    customer    TEXT,
    city        TEXT,
    pincode     TEXT,
    products    TEXT,        -- BAD: multiple values in one cell!
    dept_name   TEXT,        -- redundant with dept_id
    dept_budget REAL         -- depends on dept, not on order!
)
""")

run("""
INSERT INTO unnormalized_orders VALUES
(1, 'Alice', 'Delhi',  '110001', 'Pen, Book, Bag', 'Sales',     500000),
(2, 'Bob',   'Mumbai', '400001', 'Laptop',          'IT',        800000),
(3, 'Alice', 'Delhi',  '110001', 'Pen',             'Sales',     500000)
""")

run("SELECT * FROM unnormalized_orders")

In [0]:
# Problems with the unnormalized table:
problems = """
Problems with the unnormalized table:
======================================
1NF Violation : 'products' column has multiple values ('Pen, Book, Bag')

2NF Violation : 'dept_budget' depends on 'dept_name', not on 'order_id'
                (partial dependency in composite key scenarios)

3NF Violation : 'city' and 'pincode' depend on each other (transitive)
                Customer city/pincode should be in a customers table

Redundancy    : Alice's city/pincode is stored in EVERY row she appears
                If Alice moves, we must update MULTIPLE rows!
"""
print(problems)

In [0]:
# ── NORMALIZED version ────────────────────────────────────────────────────────

# 1NF + 2NF + 3NF — split into proper tables

run("""
CREATE TABLE customers_norm (
    customer_id INTEGER PRIMARY KEY,
    name        TEXT NOT NULL,
    city        TEXT,
    pincode     TEXT
)
""")

run("""
CREATE TABLE products_norm (
    product_id   INTEGER PRIMARY KEY,
    product_name TEXT NOT NULL,
    price        REAL
)
""")

run("""
CREATE TABLE orders_norm (
    order_id    INTEGER PRIMARY KEY,
    customer_id INTEGER REFERENCES customers_norm(customer_id),
    order_date  TEXT
)
""")

run("""
CREATE TABLE order_items_norm (
    order_id   INTEGER REFERENCES orders_norm(order_id),
    product_id INTEGER REFERENCES products_norm(product_id),
    quantity   INTEGER DEFAULT 1,
    PRIMARY KEY (order_id, product_id)
)
""")

# Insert data
cur.executescript("""
INSERT INTO customers_norm VALUES (1,'Alice','Delhi','110001');
INSERT INTO customers_norm VALUES (2,'Bob','Mumbai','400001');
INSERT INTO products_norm VALUES (1,'Pen',20);
INSERT INTO products_norm VALUES (2,'Book',150);
INSERT INTO products_norm VALUES (3,'Bag',500);
INSERT INTO products_norm VALUES (4,'Laptop',45000);
INSERT INTO orders_norm VALUES (1,1,'2024-01-15');
INSERT INTO orders_norm VALUES (2,2,'2024-01-16');
INSERT INTO orders_norm VALUES (3,1,'2024-01-20');
INSERT INTO order_items_norm VALUES (1,1,2);
INSERT INTO order_items_norm VALUES (1,2,1);
INSERT INTO order_items_norm VALUES (1,3,1);
INSERT INTO order_items_norm VALUES (2,4,1);
INSERT INTO order_items_norm VALUES (3,1,5);
""")
conn.commit()

print("✅ Normalized tables created and populated!")

# Now query — show all orders with details
run("""
SELECT
    o.order_id,
    c.name          AS customer,
    c.city,
    p.product_name,
    oi.quantity,
    p.price * oi.quantity AS total
FROM orders_norm o
JOIN customers_norm  c  ON o.customer_id  = c.customer_id
JOIN order_items_norm oi ON o.order_id    = oi.order_id
JOIN products_norm   p  ON oi.product_id  = p.product_id
ORDER BY o.order_id
""")

---
<a id='transactions'></a>
## 📌 LESSON 12 — Transactions & ACID Properties

A transaction is a group of SQL statements that execute as a **single unit** — either ALL succeed or NONE do.

### ACID Properties — Memorize This!

| Property | Meaning |
|----------|---------|
| **A**tomicity | All operations succeed, or none do. No partial completion. |
| **C**onsistency | Database moves from one valid state to another. Rules never violated. |
| **I**solation | Transactions don't interfere with each other. |
| **D**urability | Committed changes are permanent, even after system crash. |

In [0]:
# Create a bank accounts table to demonstrate transactions
run("""
CREATE TABLE bank_accounts (
    account_id   INTEGER PRIMARY KEY,
    owner        TEXT    NOT NULL,
    balance      REAL    NOT NULL CHECK(balance >= 0)
)
""")

run("INSERT INTO bank_accounts VALUES (101, 'Alice', 10000)")
run("INSERT INTO bank_accounts VALUES (202, 'Bob',   5000)")

print("Initial balances:")
run("SELECT * FROM bank_accounts")

In [0]:
# ── SUCCESSFUL TRANSACTION ────────────────────────────────────────────────────
print("Transferring ₹3000 from Alice to Bob...\n")

try:
    # Start transaction (SQLite is in autocommit by default, so we use BEGIN)
    cur.execute("BEGIN")

    cur.execute("UPDATE bank_accounts SET balance = balance - 3000 WHERE account_id = 101")
    cur.execute("UPDATE bank_accounts SET balance = balance + 3000 WHERE account_id = 202")

    conn.commit()   # ← COMMIT: save permanently
    print("✅ COMMIT — Transaction successful!")

except Exception as e:
    conn.rollback() # ← ROLLBACK: undo everything
    print(f"❌ ROLLBACK — Transaction failed: {e}")

print()
run("SELECT * FROM bank_accounts")

In [0]:
# ── FAILED TRANSACTION with ROLLBACK ─────────────────────────────────────────
print("Attempting to overdraw Alice's account (should fail and rollback)...\n")

try:
    cur.execute("BEGIN")
    cur.execute("UPDATE bank_accounts SET balance = balance - 50000 WHERE account_id = 101")
    # This next INSERT will fail because balance CHECK(balance >= 0) would be violated
    # Let's manually raise an error to simulate failure
    raise Exception("Simulated system failure mid-transaction!")
    cur.execute("UPDATE bank_accounts SET balance = balance + 50000 WHERE account_id = 202")
    conn.commit()

except Exception as e:
    conn.rollback()
    print(f"❌ ROLLBACK — Error caught: {e}")
    print("   All changes undone! Database is unchanged.")

print()
print("Balances after rollback (should be unchanged):")
run("SELECT * FROM bank_accounts")

---
<a id='views'></a>
## 📌 LESSON 13 — Views & CTEs

### Views
A **view** is a virtual table based on a stored SELECT query. No data is stored — the query runs each time you use the view.

**Benefits:** Simplicity, security (hide sensitive columns), reusability.

### CTEs (Common Table Expressions)
A **CTE** uses `WITH` to give a temporary name to a subquery. Makes complex queries readable and reusable within the same query.

In [0]:
# ── CREATE A VIEW ─────────────────────────────────────────────────────────────
run("""
CREATE VIEW employee_summary AS
SELECT
    e.employee_id,
    e.first_name || ' ' || e.last_name  AS full_name,
    e.salary,
    d.department_name,
    d.location,
    CASE
        WHEN e.salary >= 85000 THEN 'Senior'
        WHEN e.salary >= 65000 THEN 'Mid-level'
        ELSE 'Junior'
    END AS level
FROM employees e
JOIN departments d ON e.department_id = d.department_id
""")

print("✅ View 'employee_summary' created!")
print()
# Query the view just like a table
run("SELECT * FROM employee_summary ORDER BY salary DESC")

In [0]:
# Views can be filtered and sorted like tables
print("Senior employees in Engineering:")
run("""
SELECT full_name, salary
FROM employee_summary
WHERE level = 'Senior' AND department_name = 'Engineering'
""")

In [0]:
# ── CTE (Common Table Expression) ────────────────────────────────────────────
# WITH keyword — give a name to a temporary result set

print("CTE: Find departments where avg salary > company avg")
run("""
WITH company_avg AS (
    SELECT AVG(salary) AS avg_sal FROM employees
),
dept_avg AS (
    SELECT department_id, ROUND(AVG(salary), 2) AS dept_sal
    FROM employees
    GROUP BY department_id
)
SELECT
    d.department_name,
    da.dept_sal,
    ROUND(ca.avg_sal, 2)   AS company_avg,
    da.dept_sal - ca.avg_sal AS difference
FROM dept_avg da
JOIN departments d ON da.department_id = d.department_id
CROSS JOIN company_avg ca
WHERE da.dept_sal > ca.avg_sal
ORDER BY difference DESC
""")

In [0]:
# ── WINDOW FUNCTIONS ──────────────────────────────────────────────────────────
# Window functions calculate across related rows WITHOUT collapsing them

# ROW_NUMBER, RANK within each department
run("""
SELECT
    first_name,
    salary,
    department_id,
    ROW_NUMBER() OVER (PARTITION BY department_id ORDER BY salary DESC) AS rank_in_dept,
    salary - AVG(salary) OVER (PARTITION BY department_id) AS vs_dept_avg
FROM employees
ORDER BY department_id, rank_in_dept
""")

In [0]:
# Running total of salary across all employees (ordered by hire_date)
run("""
SELECT
    first_name,
    hire_date,
    salary,
    SUM(salary) OVER (ORDER BY hire_date) AS running_total_payroll
FROM employees
WHERE hire_date IS NOT NULL
ORDER BY hire_date
""")

---
<a id='coding'></a>
## 📌 LESSON 14 — Classic Interview Coding Questions

These are the **exact problems** asked in SQL interviews for fresher roles. Practice each one!

In [0]:
# ── Q1: Find the SECOND highest salary ───────────────────────────────────────

print("Method 1: Subquery")
run("""
SELECT MAX(salary) AS second_highest_salary
FROM employees
WHERE salary < (SELECT MAX(salary) FROM employees)
""")

print("\nMethod 2: LIMIT + OFFSET")
run("""
SELECT DISTINCT salary AS second_highest_salary
FROM employees
ORDER BY salary DESC
LIMIT 1 OFFSET 1
""")

In [0]:
# ── Q2: Find the Nth highest salary (general solution) ───────────────────────
# Change OFFSET N-1 for Nth highest (OFFSET 0=1st, OFFSET 2=3rd)

print("3rd highest salary:")
run("""
SELECT DISTINCT salary AS third_highest_salary
FROM employees
ORDER BY salary DESC
LIMIT 1 OFFSET 2
""")

In [0]:
# ── Q3: Find employees earning MORE than their manager ────────────────────────
run("""
SELECT
    e.first_name  AS employee,
    e.salary      AS emp_salary,
    m.first_name  AS manager,
    m.salary      AS mgr_salary
FROM employees e
JOIN employees m ON e.manager_id = m.employee_id
WHERE e.salary > m.salary
""")

In [0]:
# ── Q4: Find duplicate values in a column ────────────────────────────────────
# Add a duplicate for demo
run("INSERT INTO employees (employee_id, first_name, last_name, salary, department_id) VALUES (97, 'Arjun', 'Duplicate', 50000, 20)")
run("INSERT INTO employees (employee_id, first_name, last_name, salary, department_id) VALUES (98, 'Priya', 'Duplicate', 50000, 20)")

print("Duplicate first names:")
run("""
SELECT first_name, COUNT(*) AS occurrences
FROM employees
GROUP BY first_name
HAVING COUNT(*) > 1
""")

# Clean up
run("DELETE FROM employees WHERE employee_id IN (97, 98)")

In [0]:
# ── Q5: Department with highest average salary ───────────────────────────────
run("""
SELECT
    d.department_name,
    ROUND(AVG(e.salary), 2) AS avg_salary
FROM employees e
JOIN departments d ON e.department_id = d.department_id
GROUP BY d.department_id, d.department_name
ORDER BY avg_salary DESC
LIMIT 1
""")

In [0]:
# ── Q6: Count employees per dept INCLUDING depts with 0 employees ─────────────
run("""
SELECT
    d.department_name,
    COUNT(e.employee_id) AS employee_count
FROM departments d
LEFT JOIN employees e ON d.department_id = e.department_id
GROUP BY d.department_id, d.department_name
ORDER BY employee_count DESC
""")

In [0]:
# ── Q7: Employees NOT assigned to any project ────────────────────────────────
run("""
SELECT first_name
FROM employees
WHERE employee_id NOT IN (
    SELECT DISTINCT employee_id FROM emp_projects
)
ORDER BY first_name
""")

In [0]:
# ── Q8: Top 3 earners from EACH department ───────────────────────────────────
run("""
WITH ranked AS (
    SELECT
        first_name,
        salary,
        department_id,
        ROW_NUMBER() OVER (PARTITION BY department_id ORDER BY salary DESC) AS rn
    FROM employees
)
SELECT
    d.department_name,
    r.first_name,
    r.salary,
    r.rn AS rank_in_dept
FROM ranked r
JOIN departments d ON r.department_id = d.department_id
WHERE r.rn <= 3
ORDER BY d.department_name, r.rn
""")

In [0]:
# ── Q9: Percentage of total salary each employee represents ──────────────────
run("""
SELECT
    first_name,
    salary,
    ROUND(salary * 100.0 / (SELECT SUM(salary) FROM employees), 2) AS pct_of_total
FROM employees
ORDER BY pct_of_total DESC
""")

In [0]:
# ── Q10: Employees hired in the most recent year ─────────────────────────────
run("""
SELECT first_name, hire_date
FROM employees
WHERE SUBSTR(hire_date,1,4) = (
    SELECT MAX(SUBSTR(hire_date,1,4))
    FROM employees
    WHERE hire_date IS NOT NULL
)
ORDER BY hire_date
""")

In [0]:
# ── Q11: UNION vs UNION ALL demo ─────────────────────────────────────────────
print("UNION — removes duplicates:")
run("""
SELECT 'Engineering' AS name
UNION
SELECT 'Engineering'
UNION
SELECT 'Marketing'
""")

print("\nUNION ALL — keeps duplicates (faster!):")
run("""
SELECT 'Engineering' AS name
UNION ALL
SELECT 'Engineering'
UNION ALL
SELECT 'Marketing'
""")

In [0]:
# ── Q12: PIVOT-style query — salary by department as columns ─────────────────
run("""
SELECT
    ROUND(AVG(CASE WHEN department_id = 10 THEN salary END), 0) AS Engineering,
    ROUND(AVG(CASE WHEN department_id = 20 THEN salary END), 0) AS Marketing,
    ROUND(AVG(CASE WHEN department_id = 30 THEN salary END), 0) AS HR,
    ROUND(AVG(CASE WHEN department_id = 40 THEN salary END), 0) AS Finance,
    ROUND(AVG(CASE WHEN department_id = 50 THEN salary END), 0) AS Research
FROM employees
""")

---
<a id='qa'></a>
## 📌 LESSON 15 — Top 20 SQL Interview Questions & Answers

**Q1: What is SQL?**
Structured Query Language — the standard language for managing relational databases. Used to create, read, update, and delete data.

**Q2: What is a primary key?**
A column (or set of columns) that uniquely identifies each row. Cannot be NULL. Cannot have duplicates. Only ONE per table.

**Q3: What is a foreign key?**
A column that references the primary key of another table. Enforces referential integrity — prevents orphan records.

**Q4: Difference between CHAR and VARCHAR?**
CHAR is fixed-length — always uses the full declared size, padded with spaces. VARCHAR is variable-length — uses only the space needed. CHAR is slightly faster; VARCHAR saves storage.

**Q5: Difference between DELETE, TRUNCATE, DROP?**
- DELETE: removes specific rows, can be rolled back, fires triggers, WHERE clause optional
- TRUNCATE: removes ALL rows, faster, cannot be rolled back (usually), resets auto-increment
- DROP: removes the entire table (structure + data) permanently

**Q6: What is a JOIN? Name the types.**
Combines rows from two or more tables based on a condition. Types: INNER, LEFT, RIGHT, FULL OUTER, SELF, CROSS JOIN.

**Q7: Difference between WHERE and HAVING?**
WHERE filters individual rows BEFORE grouping. HAVING filters groups AFTER grouping. Aggregate functions (SUM, AVG, COUNT) cannot be used in WHERE but can in HAVING.

**Q8: What is a subquery?**
A query nested inside another query. The inner query runs first and its result is used by the outer query.

**Q9: What are ACID properties?**
Atomicity, Consistency, Isolation, Durability — guarantees that make database transactions reliable.

**Q10: What is normalization?**
Organizing a database to reduce data redundancy and improve integrity. Key forms: 1NF (atomic values), 2NF (no partial dependency), 3NF (no transitive dependency).

**Q11: What is an index?**
A data structure that speeds up data retrieval. Like a book index. Speeds up reads but slows down writes. Created on frequently searched columns.

**Q12: What is a view?**
A virtual table based on a stored SELECT query. Does not store data itself. Provides simplicity, security, and reusability.

**Q13: What is a stored procedure?**
A named, precompiled set of SQL statements stored in the database. Called once to execute multiple statements. Reduces repetition and network traffic.

**Q14: What is a trigger?**
SQL code that automatically executes BEFORE or AFTER an INSERT, UPDATE, or DELETE event on a table. Used for auditing, validation, cascading changes.

**Q15: What is the difference between UNION and UNION ALL?**
UNION combines results and removes duplicates (slower). UNION ALL keeps all rows including duplicates (faster).

**Q16: What is a NULL value?**
Represents the absence of a value — not zero, not empty string. Comparisons with NULL return NULL (not True/False). Always use IS NULL or IS NOT NULL.

**Q17: What is a correlated subquery?**
A subquery that references a column from the outer query. Runs once for each row of the outer query — can be slow on large tables.

**Q18: What is a CTE?**
Common Table Expression — a temporary named result set defined with WITH. Makes complex queries readable. Can be reused multiple times in the same query.

**Q19: What is a window function?**
Performs a calculation across rows related to the current row without collapsing them into groups. Examples: ROW_NUMBER(), RANK(), SUM() OVER(), AVG() OVER().

**Q20: What is denormalization?**
Intentionally adding redundancy back into a database to improve read performance. Trade-off: faster reads, harder to maintain consistency. Used in data warehouses and reporting databases.

---
<a id='tips'></a>
## 📌 LESSON 16 — SQL Execution Order & Final Tips

### 🔢 Logical Query Execution Order
This is one of the most asked interview questions!

SQL does NOT execute in the order you write it. The logical order is:

```
1. FROM          → which table(s) to use
2. JOIN          → combine tables
3. WHERE         → filter individual rows
4. GROUP BY      → group rows
5. HAVING        → filter groups
6. SELECT        → choose which columns
7. DISTINCT      → remove duplicates
8. ORDER BY      → sort the results
9. LIMIT/OFFSET  → restrict number of rows
```

This is why:
- You **cannot** use a SELECT alias in a WHERE clause (WHERE runs before SELECT)
- You **cannot** use aggregate functions in WHERE (WHERE runs before GROUP BY)
- You **can** use aggregate functions in HAVING (HAVING runs after GROUP BY)

In [0]:
# Demo: Execution order matters!

# This works — HAVING runs after GROUP BY
run("""
SELECT department_id, COUNT(*) AS cnt
FROM employees
GROUP BY department_id
HAVING COUNT(*) > 2
""")

# This would FAIL in strict SQL (alias used in WHERE — not allowed)
# SELECT salary * 12 AS annual FROM employees WHERE annual > 500000
# ← ERROR: 'annual' not recognized — WHERE runs before SELECT!

# Correct way:
run("""
SELECT first_name, salary * 12 AS annual_salary
FROM employees
WHERE salary * 12 > 500000
ORDER BY annual_salary DESC
""")

In [0]:
# ── PERFORMANCE TIPS ─────────────────────────────────────────────────────────
print("""
SQL Performance Tips (mention these in interviews!):
=====================================================

1. SELECT specific columns, not SELECT *
   → Reduces data transferred from database

2. Use indexes on columns in WHERE, JOIN, ORDER BY
   → Can turn O(n) scans into O(log n) lookups

3. Use WHERE to filter early, before JOINs if possible
   → Fewer rows to join = faster

4. Avoid functions on indexed columns in WHERE
   BAD:  WHERE YEAR(hire_date) = 2023
   GOOD: WHERE hire_date BETWEEN '2023-01-01' AND '2023-12-31'

5. Use EXISTS instead of IN for large subqueries
   → EXISTS stops as soon as it finds one match

6. Avoid SELECT DISTINCT when not necessary
   → DISTINCT requires sorting/hashing all results

7. LIMIT early in development/testing
   → Don't fetch 1 million rows while testing
""")

In [0]:
# ── FINAL CHEAT SHEET ────────────────────────────────────────────────────────
print("""
╔══════════════════════════════════════════════════════════════════╗
║         SQL INTERVIEW CHEAT SHEET — IN2IT TECHNOLOGIES          ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  5 SQL Categories: DDL, DML, DQL, DCL, TCL                      ║
║                                                                  ║
║  JOINS: INNER, LEFT, RIGHT, FULL OUTER, SELF, CROSS             ║
║         INNER = matching only                                    ║
║         LEFT  = all from left + matches from right              ║
║                                                                  ║
║  WHERE vs HAVING:                                                ║
║         WHERE  → filters ROWS (before GROUP BY)                 ║
║         HAVING → filters GROUPS (after GROUP BY)                ║
║                                                                  ║
║  DELETE vs TRUNCATE vs DROP:                                     ║
║         DELETE   → specific rows, rollback-able                 ║
║         TRUNCATE → all rows, faster, no rollback                ║
║         DROP     → entire table gone forever                    ║
║                                                                  ║
║  ACID: Atomicity, Consistency, Isolation, Durability            ║
║                                                                  ║
║  Execution Order:                                                ║
║  FROM → JOIN → WHERE → GROUP BY → HAVING → SELECT → ORDER → LIMIT║
║                                                                  ║
║  Normalization:                                                  ║
║  1NF = atomic values                                             ║
║  2NF = no partial dependency                                     ║
║  3NF = no transitive dependency                                  ║
║                                                                  ║
╠══════════════════════════════════════════════════════════════════╣
║   🎯 Interview: March 12th | 📍 Noida | 💪 You've got this!    ║
╚══════════════════════════════════════════════════════════════════╝
""")

---

## 🌟 You're Ready!

This notebook covered everything needed for a SQL fresher interview:

✅ All 5 SQL command categories with examples
✅ Full SELECT with WHERE, ORDER BY, LIMIT, DISTINCT
✅ Aggregate functions — COUNT, SUM, AVG, MAX, MIN
✅ GROUP BY and HAVING with real data
✅ All JOIN types with live queries
✅ Subqueries — scalar, correlated, inline views
✅ String, Math, Date functions and CASE WHEN
✅ Keys, Constraints, and Indexes
✅ Normalization — 1NF, 2NF, 3NF with examples
✅ Transactions and ACID properties
✅ Views and CTEs
✅ Window functions — ROW_NUMBER, running totals
✅ 12 classic interview coding problems solved
✅ Top 20 interview Q&A
✅ SQL execution order and performance tips

---
*Best of luck on March 12th! 🎯*